# Generate Synthetic NYC Taxi Data

This notebook generates realistic synthetic NYC taxi trip data for testing and development purposes.

In [ ]:
# Install necessary libraries
!pip install -q numpy pandas

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import datetime as dt
import hashlib
from pathlib import Path

In [ ]:
# Function to generate synthetic taxi data
def generate_synthetic_taxi(n=3000, seed=20250115, day="2025-01-15"):
    """
    Generate synthetic NYC taxi trip data.
    
    Args:
        n: Number of trips to generate
        seed: Random seed for reproducibility
        day: Date for the trips (ISO format)
    
    Returns:
        DataFrame with synthetic taxi trip data
    """
    rng = np.random.default_rng(seed)
    date = dt.date.fromisoformat(day)

    # Hour-of-day demand weights (morning + evening peaks, low overnight)
    hours = np.arange(24)
    weights = np.array([
        0.020, 0.015, 0.012, 0.010, 0.012, 0.020,
        0.040, 0.060, 0.080, 0.060, 0.050, 0.045,
        0.045, 0.050, 0.055, 0.060, 0.080, 0.090,
        0.075, 0.060, 0.050, 0.040, 0.030, 0.025
    ])
    weights = weights / weights.sum()

    # Allocate pickups per hour
    counts = rng.multinomial(n, weights)
    for h in range(24):
        if counts[h] == 0:
            donor = int(np.argmax(counts))
            counts[donor] -= 1
            counts[h] = 1

    # Generate pickup timestamps
    pickup_times = []
    for h, c in enumerate(counts):
        minute = rng.integers(0, 60, size=c)
        second = rng.integers(0, 60, size=c)
        mask = rng.random(c) < 0.12
        minute[mask] = rng.choice([0, 15, 30, 45], size=mask.sum())
        for m, s in zip(minute, second):
            pickup_times.append(dt.datetime.combine(date, dt.time(hour=int(h), minute=int(m), second=int(s))))

    pickup_times = np.array(pickup_times, dtype="datetime64[s]")
    rng.shuffle(pickup_times)

    # Passenger counts (mostly 1-2)
    passenger = rng.choice([1,2,3,4,5,6], size=n, p=[0.62,0.23,0.08,0.04,0.02,0.01]).astype(float)

    # Payment types
    payment_types = rng.choice([1,2,3,4,5,6,0], size=n, p=[0.68,0.26,0.01,0.01,0.02,0.01,0.01])

    # Trip distance (miles)
    dist = rng.lognormal(mean=0.9, sigma=0.65, size=n)
    dist = np.clip(dist, 0.01, 50.0)
    long_idx = rng.choice(n, size=max(1, int(0.015*n)), replace=False)
    dist[long_idx] = rng.uniform(20, 50, size=len(long_idx))
    short_pool = np.setdiff1d(np.arange(n), long_idx)
    short_idx = rng.choice(short_pool, size=max(1, int(0.02*n)), replace=False)
    dist[short_idx] = rng.uniform(0.01, 0.12, size=len(short_idx))

    # Speed (mph)
    pickup_dt = pickup_times.astype("datetime64[s]").astype(object)
    hours_of_day = np.array([t.hour for t in pickup_dt], dtype=int)
    speed = rng.normal(loc=16, scale=4, size=n)
    rush = ((hours_of_day >= 7) & (hours_of_day <= 9)) | ((hours_of_day >= 16) & (hours_of_day <= 19))
    speed[rush] -= rng.normal(loc=3, scale=1.5, size=rush.sum())
    speed = np.clip(speed, 5, 30)

    # Duration
    duration = dist / speed * 60.0
    duration += rng.normal(loc=2.5, scale=3.0, size=n)
    duration = np.clip(duration, 0.2, 240.0)
    jam_idx = rng.choice(n, size=max(1, int(0.01*n)), replace=False)
    duration[jam_idx] *= rng.uniform(1.8, 3.0, size=len(jam_idx))
    duration = np.clip(duration, 0.2, 240.0)

    # Fare
    fare = 3.0 + dist*2.4 + duration*0.35 + rng.normal(0, 1.25, size=n)
    fare = np.clip(fare, 2.5, 400.0)

    # Tolls
    tolls = np.zeros(n)
    toll_mask = rng.random(n) < 0.12
    tolls[toll_mask] = rng.choice([6.94, 8.25, 12.50, 0.00], size=toll_mask.sum(), p=[0.55,0.15,0.20,0.10])

    # Tips
    tip = np.zeros(n)
    cc = payment_types == 1
    tip_rate = rng.beta(2.2, 7.5, size=cc.sum()) * 0.40
    zero_tip_mask = rng.random(cc.sum()) < 0.08
    tip_rate[zero_tip_mask] = 0.0
    tip[cc] = fare[cc] * tip_rate

    # Hidden charges
    hidden = rng.choice([0.0, 0.5, 1.0, 1.5, 2.5, 3.0, 4.5], size=n, p=[0.05,0.16,0.22,0.20,0.18,0.12,0.07])

    fare = np.round(fare, 2)
    tip = np.round(tip, 2)
    tolls = np.round(tolls, 2)
    total = np.round(fare + tip + tolls + hidden, 2)

    # Dropoff timestamps
    pickup_py = np.array([pd.Timestamp(t).to_pydatetime() for t in pickup_dt])
    dropoff_py = pickup_py + np.array([dt.timedelta(minutes=float(x)) for x in duration])
    end_of_day = dt.datetime.combine(date, dt.time(23,59,59))
    dropoff_py = np.array([min(t, end_of_day) for t in dropoff_py], dtype=object)

    # Add data quality issues for testing
    idx_all = np.arange(n)
    missing_n = max(1, int(0.008*n))
    idx_pickup_missing = rng.choice(idx_all, size=missing_n, replace=False)
    remaining = np.setdiff1d(idx_all, idx_pickup_missing)
    idx_dropoff_missing = rng.choice(remaining, size=missing_n, replace=False)
    remaining = np.setdiff1d(remaining, idx_dropoff_missing)
    idx_dist_missing = rng.choice(remaining, size=missing_n, replace=False)
    remaining = np.setdiff1d(remaining, idx_dist_missing)
    idx_total_missing = rng.choice(remaining, size=missing_n, replace=False)

    # Negative duration anomalies
    neg_n = max(1, int(0.005*n))
    remaining2 = np.setdiff1d(remaining, idx_total_missing)
    idx_neg_duration = rng.choice(remaining2, size=neg_n, replace=False)
    for i in idx_neg_duration:
        dropoff_py[i] = pickup_py[i] - dt.timedelta(minutes=float(rng.uniform(0.5, 15)))

    # Passenger anomalies
    edge_pax_idx = rng.choice(idx_all, size=max(2, int(0.004*n)), replace=False)
    passenger[edge_pax_idx[:len(edge_pax_idx)//2]] = 0
    passenger[edge_pax_idx[len(edge_pax_idx)//2:]] = 10

    # Total anomalies
    high_total_idx = rng.choice(idx_all, size=max(1, int(0.003*n)), replace=False)
    total[high_total_idx] = 999.99

    # Apply missing fields
    pickup_str = np.array([t.strftime("%Y-%m-%d %H:%M:%S") for t in pickup_py], dtype=object)
    dropoff_str = np.array([t.strftime("%Y-%m-%d %H:%M:%S") for t in dropoff_py], dtype=object)
    pickup_str[idx_pickup_missing] = ""
    dropoff_str[idx_dropoff_missing] = ""
    dist = np.round(dist, 2)
    dist[idx_dist_missing] = np.nan
    total[idx_total_missing] = np.nan

    # Cash tips set to 0
    tip[payment_types == 2] = 0.0

    df = pd.DataFrame({
        "tpep_pickup_datetime": pickup_str,
        "tpep_dropoff_datetime": dropoff_str,
        "passenger_count": pd.Series(passenger).astype("Int64"),
        "trip_distance": dist,
        "fare_amount": fare,
        "tip_amount": tip,
        "tolls_amount": tolls,
        "total_amount": total,
        "payment_type": payment_types.astype(int),
    })

    return df

print("✓ Synthetic data generation function loaded")

In [ ]:
# Validation function for data quality checks
def validate(df):
    """
    Validate the generated synthetic data and compute summary statistics.
    
    Args:
        df: DataFrame with taxi trip data
    
    Returns:
        Dictionary with validation statistics
    """
    pickup = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    dropoff = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
    duration_min = (dropoff - pickup).dt.total_seconds() / 60.0

    missing_required = pickup.isna() | dropoff.isna() | df["trip_distance"].isna() | df["total_amount"].isna()

    stats = {
        "total_rows": int(len(df)),
        "rows_missing_required": int(missing_required.sum()),
        "missing_pickup": int(pickup.isna().sum()),
        "missing_dropoff": int(dropoff.isna().sum()),
        "missing_trip_distance": int(df["trip_distance"].isna().sum()),
        "missing_total_amount": int(df["total_amount"].isna().sum()),
        "trip_distance_min": float(df["trip_distance"].min(skipna=True)),
        "trip_distance_max": float(df["trip_distance"].max(skipna=True)),
        "trip_distance_mean": float(df["trip_distance"].mean(skipna=True)),
        "trip_duration_min_min": float(duration_min.min(skipna=True)),
        "trip_duration_min_max": float(duration_min.max(skipna=True)),
        "trip_duration_min_mean": float(duration_min.mean(skipna=True)),
        "total_amount_min": float(df["total_amount"].min(skipna=True)),
        "total_amount_max": float(df["total_amount"].max(skipna=True)),
        "total_amount_mean": float(df["total_amount"].mean(skipna=True)),
        "payment_type_counts": df["payment_type"].value_counts(dropna=False).sort_index().to_dict(),
    }
    return stats

print("✓ Validation function loaded")

In [ ]:
# Generate synthetic data
print("Generating synthetic taxi data...")
df = generate_synthetic_taxi(n=3000, seed=20250115, day="2025-01-15")
print(f"✓ Generated {len(df)} trips")

In [ ]:
# Save to raw_data directory
import os

# Create raw_data directory in the same location as the notebook
output_dir = Path("raw_data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "yellow_tripdata_sample.csv"
df.to_csv(output_file, index=False)

# Calculate file hash for verification
csv_bytes = output_file.read_bytes()
file_hash = hashlib.sha256(csv_bytes).hexdigest()

# Show both relative and absolute paths
abs_path = output_file.resolve()

print(f"✓ Saved to: {output_file}")
print(f"  Absolute path: {abs_path}")
print(f"  Current working directory: {os.getcwd()}")
print(f"  File size: {len(csv_bytes):,} bytes")
print(f"  SHA256: {file_hash}")
print(f"\nFile exists: {output_file.exists()}")

In [ ]:
# Validate and display summary statistics
summary = validate(df)
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)
print(f"Total rows: {summary['total_rows']:,}")
print(f"Rows with missing required fields: {summary['rows_missing_required']}")
print(f"\nMissing values:")
print(f"  - Pickup datetime: {summary['missing_pickup']}")
print(f"  - Dropoff datetime: {summary['missing_dropoff']}")
print(f"  - Trip distance: {summary['missing_trip_distance']}")
print(f"  - Total amount: {summary['missing_total_amount']}")
print(f"\nTrip distance (miles):")
print(f"  - Min: {summary['trip_distance_min']:.2f}")
print(f"  - Max: {summary['trip_distance_max']:.2f}")
print(f"  - Mean: {summary['trip_distance_mean']:.2f}")
print(f"\nTrip duration (minutes):")
print(f"  - Min: {summary['trip_duration_min_min']:.2f}")
print(f"  - Max: {summary['trip_duration_min_max']:.2f}")
print(f"  - Mean: {summary['trip_duration_min_mean']:.2f}")
print(f"\nTotal amount ($):")
print(f"  - Min: {summary['total_amount_min']:.2f}")
print(f"  - Max: {summary['total_amount_max']:.2f}")
print(f"  - Mean: {summary['total_amount_mean']:.2f}")
print(f"\nPayment type distribution:")
for ptype, count in sorted(summary['payment_type_counts'].items()):
    print(f"  - Type {ptype}: {count:,} trips")
print("="*60)

# Return summary for notebook output
summary

In [ ]:
# Preview the generated data
print("\nFirst 10 rows:")
df.head(10)